# 2026 MDS Engagement with DASH — data pull

Builds the **"2026 MDS Engagement with DASH"** summary table straight from Postgres
(`rawform_forms_v3` for non-final-year cohorts, `dds4_boh3_forms_v3` for final-year
BOH3/DDS4), for the nine subject codes in the report, up to a reporting cutoff date.

**Data source paths** follow the two rawforms-v3 handover docs
(`boh2_dds2_dds3_cohort_reports_v3_handover.md`, `rawforms_v3_separation_handover.md`):
scales live at `assessor_data->'scales'->'<key>'->>'key'` (nested, prefixed keys),
`role`/`clinic`/`patient_data` come from `form_context`, final-year `patient_data` is a JSON **array**.

Run top to bottom. The output workbook is written next to the notebook and also displayed inline.

> **Every metric definition is spelled out in the CONFIG cell and in `SQL` comments.**
> Several rows are genuinely ambiguous in the template (see the **ASSUMPTIONS** cell) —
> tweak the config dicts / thresholds there rather than editing SQL.

In [ ]:
# ============================ CONFIG ============================
import os, re
import pandas as pd
from datetime import date
from sqlalchemy import create_engine, text

# --- DB connection (reads .env in this folder; same creds as main.ipynb) ---
# .env: DB_USER / DB_PASSWORD / DB_HOST / DB_PORT / DB_NAME
def _load_env(path=".env"):
    env = {}
    if os.path.exists(path):
        for line in open(path):
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            env[k.strip()] = v.strip().strip('"').strip("'")
    return env

_env = _load_env()
DB_USER = _env.get("DB_USER", "postgres")
DB_PASSWORD = _env.get("DB_PASSWORD", "")
DB_HOST = _env.get("DB_HOST", "localhost")
DB_PORT = _env.get("DB_PORT", "5432")
DB_NAME = _env.get("DB_NAME", "MDS_DASH")

# psycopg (v3) is what general_utils imports; change to +psycopg2 if you use that driver.
ENGINE_URL = f"postgresql+psycopg://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(ENGINE_URL)
# If you already have an `engine` in your session, just reuse it and skip the two lines above.

def readDf(engine, sql, params=None):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params or {})

# --- Reporting cutoff (report says "Reported up to the 5th August 2026") ---
CUTOFF = date(2026, 8, 5)          # inclusive; filters on datetimeutc::date <= CUTOFF

# --- The nine subject codes, in the report's column order ---
SUBJECTS = ["ORAL10005","ORAL20005","ORAL30002","DENT90141","DENT90146",
            "DENT90148","DENT90150","DENT90151","DENT90124"]

# --- clinic_type code -> named discipline (report's six discipline rows) ---
# Codes observed in the data so far: GP, PE, EN, CD, RP(?), FP(?), PA/PD(?).
# Anything not mapped here is bucketed under "Other" (reported but not in the six rows).
DISCIPLINE_MAP = {
    "GP": "General Practice",
    "PE": "Periodontics",
    "RP": "Removable Pros",
    "EN": "Endodontics",
    "FP": "Fixed Pros",
    "PA": "Paediatrics",
    "PD": "Paediatrics",
}
DISCIPLINE_ORDER = ["General Practice","Periodontics","Removable Pros",
                    "Endodontics","Fixed Pros","Paediatrics"]

# --- role code meaning "Operator" (form_context.role) ---
OPERATOR_ROLE = "O"

# --- A "session" = one student attending one clinic/sim block on one day in one discipline ---
#     i.e. distinct (student_number, calendar date, clinic).  Set SESSION_AS_FORMS=True
#     to instead count raw forms (1 form = 1 session).
SESSION_AS_FORMS = False

# --- "Not meeting the standard" flag (PLACEHOLDER - align to your real flagging rule) ---
#     A student is flagged if ANY of their forms has global-rating in FLAG_GR_LEVELS
#     or practice-readiness/entrustment level in FLAG_PR_LEVELS.
FLAG_GR_LEVELS = [1, 2]      # global rating below "Satisfactory" (3)
FLAG_PR_LEVELS = [1]         # practice-readiness / entrustment level 1
print("Config loaded. Cutoff:", CUTOFF, "| Subjects:", len(SUBJECTS))

## ASSUMPTIONS — read before trusting the numbers

Each row of the template maps to a definition below. Where the template wording is
ambiguous the choice is called out; change it in **CONFIG** or the relevant SQL.

| Template row | Definition used |
|---|---|
| Total number of student forms overall | Count of individual assessment **forms** (exploded form objects), all types |
| Total number of assessors | Distinct `assessor_email` |
| Total Clinical Assessment Forms (incl Sim & Observations) | All forms (type ∈ Simulation, Clinic, Observation) — equals row 1 here |
| Total Clinic Assessment Forms | `type = 'Clinic'` |
| Total Sim Assessment Forms | `type = 'Simulation'` |
| Operator sessions per student | Distinct (student, date, clinic) where `role = 'O'` (`SESSION_AS_FORMS` toggles to raw form count) |
| Sim sessions per student | Distinct (student, date, clinic) where `type = 'Simulation'` |
| Clinical specific sessions (+ 6 disciplines) | Clinic sessions split by `clinic_type` → `DISCIPLINE_MAP` |
| Sim specific sessions (+ 6 disciplines) | Sim sessions split by `clinic_type` → `DISCIPLINE_MAP` |
| Total Patients Seen | Forms whose `patient` has real info and is **not** an FTA (final-year: patients with `patient_attended=true`) |
| FTAs/CAs | Forms where `patient.fta_substitute` is set (final-year: `patient_attended=false`) |
| Clinical Incidences | Forms with `assessor_data.radio.'clinical-incident-occurred' = 'yes'` |
| Students flagged as not meeting the standard | **PLACEHOLDER** — distinct students with any form at `FLAG_GR_LEVELS`/`FLAG_PR_LEVELS`; swap in your `getFlaggedFormDetails` rule |
| Assessments included in the platform | Distinct `assessmentid` |

**Final-year subjects (BOH3 `ORAL30002`, and any DENT that lives in `dds4_boh3_forms_v3`)**
have no `role`/`clinic_type`, so the Operator/Sim and per-discipline rows are left blank
for them; patients/incidents/flags are computed from the array-shaped `patient_data`.

**DENT90141 local ground-truth** (from `temp 2026 caf.json`, cutoff ≤ 2026-08-05):
966 forms · 38 assessors · 5 clinic · 961 sim · 2 FTAs · 16 incidents · 806 assessmentids.
Use these to confirm the SQL path matches your DB before circulating the workbook.

In [ ]:
# ============ Which table holds each subject? ============
# Non-final-year -> rawform_forms_v3 ; final-year (BOH3/DDS4) -> dds4_boh3_forms_v3.
# Auto-detect by presence so you don't have to hardcode the DDS4/BOH3 subject codes.
def subject_table(engine, subject):
    q = "SELECT %s AS n"
    in_main = readDf(engine,
        "SELECT count(*) AS n FROM rawform_forms_v3 WHERE subject=:s", {"s": subject}).n[0]
    in_fy = readDf(engine,
        "SELECT count(*) AS n FROM dds4_boh3_forms_v3 WHERE subject=:s", {"s": subject}).n[0]
    if in_main > 0:
        return "rawform_forms_v3", "standard"
    if in_fy > 0:
        return "dds4_boh3_forms_v3", "finalyear"
    return None, "missing"

ROUTING = {s: subject_table(engine, s) for s in SUBJECTS}
for s, (tbl, kind) in ROUTING.items():
    print(f"{s:11s} -> {kind:9s} ({tbl})")

In [ ]:
# ============ Metrics for NON-final-year subjects (rawform_forms_v3) ============
_SESSION_EXPR = ("1"  # placeholder, replaced below
                 )

def _std_scalar_sql(subject):
    # single-row scalar metrics
    session_col = "form_code" if SESSION_AS_FORMS else \
        "student_number || '|' || datetimeutc::date || '|' || coalesce(clinic,'')"
    return f"""
    WITH base AS (
        SELECT *,
               regexp_replace(coalesce(assessor_data->'scales'->'scale-global-rating'->>'key',''),'\\D','','g') AS gr_s,
               regexp_replace(coalesce(assessor_data->'scales'->'scale-practice-readiness'->>'key',''),'\\D','','g') AS pr_s
        FROM rawform_forms_v3
        WHERE subject = :subject AND datetimeutc::date <= :cutoff
    )
    SELECT
        count(*)                                                            AS total_forms,
        count(DISTINCT assessor_email)                                      AS assessors,
        count(*) FILTER (WHERE type = 'Clinic')                             AS clinic_forms,
        count(*) FILTER (WHERE type = 'Simulation')                         AS sim_forms,
        count(*) FILTER (WHERE type IN ('Simulation','Clinic','Observation')) AS clinical_all,
        count(DISTINCT {session_col}) FILTER (WHERE role = :oprole)         AS operator_sessions,
        count(DISTINCT {session_col}) FILTER (WHERE type = 'Simulation')    AS sim_sessions,
        count(*) FILTER (
            WHERE nullif(patient_data->>'fta_substitute','') IS NULL
              AND ( nullif(patient_data->>'age','')  IS NOT NULL
                 OR nullif(patient_data->>'drn','')  IS NOT NULL
                 OR nullif(patient_data->>'details','') IS NOT NULL )
        )                                                                   AS patients_seen,
        count(*) FILTER (WHERE nullif(patient_data->>'fta_substitute','') IS NOT NULL) AS ftas_cas,
        count(*) FILTER (WHERE lower(assessor_data->'radio'->>'clinical-incident-occurred') = 'yes') AS incidents,
        count(DISTINCT student_number) FILTER (
            WHERE (gr_s <> '' AND gr_s::int = ANY(:flag_gr))
               OR (pr_s <> '' AND pr_s::int = ANY(:flag_pr))
        )                                                                   AS flagged_students,
        count(DISTINCT assessmentid)                                        AS assessments_platform
    FROM base
    """

def _std_discipline_sql(subject):
    # distinct sessions per (type, clinic) — split into clinic vs sim in pandas
    session_col = "form_code" if SESSION_AS_FORMS else \
        "student_number || '|' || datetimeutc::date || '|' || coalesce(clinic,'')"
    return f"""
        SELECT type, clinic,
               count(DISTINCT {session_col}) AS n
        FROM rawform_forms_v3
        WHERE subject = :subject AND datetimeutc::date <= :cutoff
        GROUP BY type, clinic
    """

def standard_metrics(engine, subject):
    p = {"subject": subject, "cutoff": CUTOFF, "oprole": OPERATOR_ROLE,
         "flag_gr": FLAG_GR_LEVELS, "flag_pr": FLAG_PR_LEVELS}
    row = readDf(engine, _std_scalar_sql(subject), p).iloc[0].to_dict()
    disc = readDf(engine, _std_discipline_sql(subject), p)
    return row, disc

In [ ]:
# ============ Metrics for FINAL-YEAR subjects (dds4_boh3_forms_v3) ============
# patient_data is a JSON ARRAY; no role / clinic_type. Operator/Sim/discipline rows blank.
def finalyear_metrics(engine, subject):
    p = {"subject": subject, "cutoff": CUTOFF, "flag_e": ['S%d'%l for l in FLAG_PR_LEVELS]}
    sql = """
    WITH base AS (
        SELECT * FROM dds4_boh3_forms_v3
        WHERE subject = :subject AND datetimeutc::date <= :cutoff
    ),
    pat AS (
        SELECT b.form_code, b.assessmentid,
               (pe->>'patient_attended')::boolean AS attended
        FROM base b,
             LATERAL jsonb_array_elements(
                 CASE WHEN jsonb_typeof(patient_data)='array' THEN patient_data ELSE '[]'::jsonb END
             ) pe
    )
    SELECT
        (SELECT count(*) FROM base)                                        AS total_forms,
        (SELECT count(DISTINCT assessor_email) FROM base)                  AS assessors,
        (SELECT count(*) FILTER (WHERE type = 'Clinic') FROM base)         AS clinic_forms,
        (SELECT count(*) FILTER (WHERE type = 'Simulation') FROM base)     AS sim_forms,
        (SELECT count(*) FROM base)                                        AS clinical_all,
        (SELECT count(*) FILTER (WHERE attended IS TRUE)  FROM pat)        AS patients_seen,
        (SELECT count(*) FILTER (WHERE attended IS FALSE) FROM pat)        AS ftas_cas,
        (SELECT count(*) FILTER (
            WHERE lower(assessor_data->'radio'->>'clinical-incident-occurred') = 'yes'
               OR jsonb_array_length(coalesce(assessor_data->'multi-select'->'clinical-incident','[]'::jsonb)) > 0
         ) FROM base)                                                      AS incidents,
        (SELECT count(DISTINCT student_number) FILTER (
            WHERE assessor_data->'scales'->'scale-entrustment'->>'key' = ANY(:flag_e)
         ) FROM base)                                                      AS flagged_students,
        (SELECT count(DISTINCT assessmentid) FROM base)                    AS assessments_platform
    """
    row = readDf(engine, sql, p).iloc[0].to_dict()
    # rows that don't apply to final-year
    row["operator_sessions"] = None
    row["sim_sessions"] = None
    return row, None

In [ ]:
# ============ Assemble the report-shaped table ============
ROW_LABELS = [
    "Total number of student forms overall",
    "Total number of assessors ",
    "Total Number of Clinical Assessment Forms (include Sim & Observations)",
    "Total Number of Clinic Assessment Forms",
    "Total Number of Sim Assessment Forms",
    "Total number of Operator sessions per student",
    "Total number of Sim sessions per student",
    "Total number of clinical specific sessions",
    "* General Practice",
    "* Periodontics",
    "* Removable Pros",
    "* Endodontics",
    "* Fixed Pros",
    "* Paediatrics",
    "Total number of sim specific sessions",
    "* General Practice",
    "* Periodontics",
    "* Removable Pros",
    "* Endodontics",
    "* Fixed Pros",
    "* Paediatrics",
    "Total Number of Patients Seen (where possible)",
    "Total number of FTAs/CAs",
    "Number of Clinical Incidences",
    "Number of students flagged as not meeting the standard (if possible)",
    "Assessments included in the platform",
]

def disc_counts(disc_df, want_type):
    """Return {discipline_name: n} for a given type, mapping clinic codes -> names."""
    out = {d: 0 for d in DISCIPLINE_ORDER}
    other = 0
    if disc_df is None:
        return out, None
    sub = disc_df[disc_df["type"] == want_type]
    for _, r in sub.iterrows():
        name = DISCIPLINE_MAP.get((r["clinic"] or "").strip())
        if name in out:
            out[name] += int(r["n"])
        else:
            other += int(r["n"])
    return out, other

def build_column(subject):
    tbl, kind = ROUTING[subject]
    if kind == "missing":
        return {lbl: None for lbl in ROW_LABELS}, {}
    if kind == "standard":
        row, disc = standard_metrics(engine, subject)
    else:
        row, disc = finalyear_metrics(engine, subject)

    cl_disc, cl_other = disc_counts(disc, "Clinic")
    sim_disc, sim_other = disc_counts(disc, "Simulation")
    clinic_sessions_total = (sum(cl_disc.values()) + (cl_other or 0)) if disc is not None else None
    sim_sessions_total = row.get("sim_sessions")

    def g(k):
        v = row.get(k)
        return int(v) if v is not None and pd.notna(v) else v

    col = {
        "Total number of student forms overall": g("total_forms"),
        "Total number of assessors ": g("assessors"),
        "Total Number of Clinical Assessment Forms (include Sim & Observations)": g("clinical_all"),
        "Total Number of Clinic Assessment Forms": g("clinic_forms"),
        "Total Number of Sim Assessment Forms": g("sim_forms"),
        "Total number of Operator sessions per student": g("operator_sessions"),
        "Total number of Sim sessions per student": g("sim_sessions"),
        "Total number of clinical specific sessions": clinic_sessions_total,
        "Total number of sim specific sessions": (sum(sim_disc.values()) + (sim_other or 0)) if disc is not None else None,
        "Total Number of Patients Seen (where possible)": g("patients_seen"),
        "Total number of FTAs/CAs": g("ftas_cas"),
        "Number of Clinical Incidences": g("incidents"),
        "Number of students flagged as not meeting the standard (if possible)": g("flagged_students"),
        "Assessments included in the platform": g("assessments_platform"),
    }
    # discipline rows appear twice (clinical block, then sim block) -> fill positionally later
    extras = {"clinic_disc": cl_disc, "sim_disc": sim_disc,
              "clinic_other": cl_other, "sim_other": sim_other}
    return col, extras

columns, extras_all = {}, {}
for s in SUBJECTS:
    columns[s], extras_all[s] = build_column(s)

# Build the DataFrame row by row, handling the two discipline blocks positionally
data = {s: [] for s in SUBJECTS}
in_sim_block = False
for i, lbl in enumerate(ROW_LABELS):
    if lbl == "Total number of clinical specific sessions":
        in_sim_block = False
    if lbl == "Total number of sim specific sessions":
        in_sim_block = True
    for s in SUBJECTS:
        if lbl.startswith("* "):
            d = lbl[2:]
            block = extras_all[s].get("sim_disc" if in_sim_block else "clinic_disc") or {}
            data[s].append(block.get(d))
        else:
            data[s].append(columns[s].get(lbl))

report = pd.DataFrame(data, index=ROW_LABELS)
report.index.name = "2026 MDS Engagement with DASH"
report

In [ ]:
# ============ Write the Excel workbook ============
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

OUT = "2026_MDS_Engagement_with_DASH.xlsx"
NAVY = "010D44"

# Second header row from the template (assessment platform per subject)
PLATFORM_NOTE = {
    "DENT90141": "FHY - Feedback Fruits\nSHY - DASH",
    "DENT90146": "DASH - SIM\nDASH - Clinic",
}
DEFAULT_NOTE = "DASH"

wb = Workbook(); ws = wb.active; ws.title = "Engagement"
ws["A1"] = "2026 MDS Engagement with DASH"
ws["A2"] = f"Reported up to the {CUTOFF.strftime('%d %B %Y')}"
ws["A1"].font = Font(bold=True, size=13); ws["A2"].font = Font(italic=True)

hdr_row = 4
ws.cell(hdr_row, 1, "").font = Font(bold=True)
for j, s in enumerate(SUBJECTS, start=2):
    c = ws.cell(hdr_row, j, s)
    c.font = Font(bold=True, color="FFFFFF"); c.fill = PatternFill("solid", fgColor=NAVY)
    c.alignment = Alignment(horizontal="center")
    n = ws.cell(hdr_row + 1, j, PLATFORM_NOTE.get(s, DEFAULT_NOTE))
    n.alignment = Alignment(horizontal="center", wrap_text=True); n.font = Font(size=9, italic=True)
ws.cell(hdr_row + 1, 1, "Platform").font = Font(italic=True, size=9)

start = hdr_row + 2
for i, lbl in enumerate(ROW_LABELS):
    r = start + i
    lc = ws.cell(r, 1, lbl)
    lc.font = Font(bold=not lbl.startswith("* "))
    if lbl.startswith("* "):
        lc.alignment = Alignment(indent=1)
    for j, s in enumerate(SUBJECTS, start=2):
        v = report.iloc[i][s]          # positional: row labels repeat across the two blocks
        ws.cell(r, j, ("" if v is None or (isinstance(v, float) and pd.isna(v)) else v)).alignment = \
            Alignment(horizontal="center")

# widths + borders
ws.column_dimensions["A"].width = 62
for j in range(2, 2 + len(SUBJECTS)):
    ws.column_dimensions[chr(64 + j) if j <= 26 else "A" + chr(64 + j - 26)].width = 12
thin = Side(style="thin", color="D9D9D9")
for row in ws.iter_rows(min_row=hdr_row, max_row=start + len(ROW_LABELS) - 1,
                        min_col=1, max_col=1 + len(SUBJECTS)):
    for c in row:
        c.border = Border(thin, thin, thin, thin)
ws.freeze_panes = "B6"
wb.save(OUT)
print("Wrote", OUT)